# 3-9 map与replace映射替换

`map()` 和 `replace()` 都能根据规则转换值，但对于规则没有覆盖到的值，两者的处理不同。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备数据路径

本节读取本章 `data` 文件夹中的模拟练习数据，使用 `pathlib` 处理相对路径。

In [ ]:
from pathlib import Path

# 从本章目录运行时使用 data；从项目根目录运行时定位到本章。
data_dir = Path("data")
if not data_dir.is_dir():
    data_dir = Path("第3章-Pandas") / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("未找到本章的 data 文件夹。")

data_dir.as_posix()

In [ ]:
data = pd.read_csv(data_dir / "map.csv", encoding="utf-8")
data.head()

## 2. 使用 map 映射单列

字典的键表示原值、值表示转换后的值。本例将性别的文字转换为数字，用于演示类别编码。

In [ ]:
gender_code = data["gender"].map({"男": 1, "女": 0})
pd.DataFrame({"原值": data["gender"], "编码": gender_code}).head()

文件第一行的性别是 `-`，字典中没有该键，所以映射结果为 `NaN`。`map()` 不会自动保留未知类别。

In [ ]:
data.loc[gender_code.isna(), ["gender", "age"]].head()

### 2.1 使用函数映射

除了字典，也可以传入函数，让函数作用于每一个元素。`na_action="ignore"` 跳过已有缺失值。

In [ ]:
s = pd.Series(["cat", "dog", np.nan, "rabbit"])
s.map("i am a {}".format, na_action="ignore")

结果中的空值仍为空值，而不会被格式化成一段描述。

## 3. 使用 replace 替换值

`replace()` 只替换规则命中的值；其他值保持原状。

In [ ]:
df = pd.DataFrame({
    "名称": ["产品1", "产品2", "产品3", "产品4", "产品5"],
    "金额": [0, 0.48, "F", 0.74, 0],
    "合计": [0.37, 0.28, "C", 0.57, 0.06]
})
df

### 3.1 替换指定列

把“金额”列中的 `F` 替换为字符串 `"0"`，再用 `pd.to_numeric()` 明确转换成数值，并将结果赋回整列。这样不依赖旧版本的自动类型推断。

In [ ]:
df["金额"] = pd.to_numeric(df["金额"].replace({"F": "0"}))
df

避免使用 `df.金额.replace(..., inplace=True)` 对临时列对象进行原地修改；使用 `df["金额"] = ...` 明确将结果赋回整列。

### 3.2 在整张表中替换

不带列选择的 `df.replace()` 会检查整张表。默认返回新对象。下例先把 `C` 换成字符串 `"2"`，稍后统一转换为数值。

In [ ]:
replaced = df.replace("C", "2")
replaced

In [ ]:
df

原表的“合计”仍然保留 `C`。若想保留替换后的表，需要写成 `df = df.replace(...)`。混合文本列清洗后，若要参与数值计算，可以再使用 `pd.to_numeric()` 明确转换。

In [ ]:
replaced["金额"] = pd.to_numeric(replaced["金额"])
replaced["合计"] = pd.to_numeric(replaced["合计"])
replaced[["金额", "合计"]].dtypes

## 4. 对比 map 与 replace

| 对比项 | `Series.map()` | `replace()` |
|---|---|---|
| 常见用途 | 编码、映射、逐元素转换 | 修改部分已有值 |
| 未在字典中出现的值 | 变为缺失值 | 保留原值 |
| 作用对象 | 本节使用 Series | Series 或 DataFrame |

In [ ]:
example = pd.Series(["男", "女", "未知"])
pd.DataFrame({"原值": example, "map结果": example.map({"男": 1, "女": 0}), "replace结果": example.replace({"男": 1, "女": 0})})

## 5. 要点总结

- 字典 `map` 未匹配的值变为缺失值；`replace` 未匹配的值保持不变。
- `map` 可接收函数，`na_action="ignore"` 跳过缺失值。
- 替换某列后用 `df[列名] = ...` 保存结果，避免对临时列使用 `inplace=True`。